# SC4001 Phase 4, DAVIS mutant stress test: training on `split_mutant`

Accelerator must be **GPU (T4 x2 or P100)**. `src/train.py` refuses CPU by design.
Code and data come from the attached dataset `isaacwjk21/sc4001-project`; no internet or installs.

## Why this kernel exists

DAVIS carries real kinase point mutants with measured affinities, so a wild-type/mutant pair
measured against the same ligand is a question with ground truth: the measured affinity shift
is known and the model's predicted shift can be scored against it. 44 usable mutants give 2992
comparable triples.

The plan had been that this needed no new training, because checkpoints already exist. That
was wrong. **Mutants share a gene with their wild type, and the cold-target split is by gene,
so under `split_cold_target` all 2992 mutant pairs fall in train (2244) or val (748) and none
in test.** Every existing DAVIS checkpoint has therefore already seen the pairs the stress test
asks about. Scoring the shift on those checkpoints answers "can the architecture represent a
point edit", not "does it generalise to mutants it has never seen".

Both questions are worth answering and the first one already has an answer, measured locally on
the three `split_cold_target` seeds, on visible mutants with both measurements uncensored
(n = 1127): Pearson r of +0.664, +0.713 and +0.803 between predicted and measured shift, with
the ratio of predicted to measured spread at 0.53, 0.67 and 0.87. So the architecture is not
blind to point mutations, but it compresses them in every seed, and the seed spread is wide
enough that one seed alone would have misled.

This kernel answers the second question. `split_mutant` is built for it: all 2992 mutant-target
pairs are its test set, and the wild types are train (451) and val (93).

## Design

DAVIS `split_mutant`, training seeds 4001-4003, three baseline runs with `src/train.py`.
Default `patience` (15) and the default output root, matching every other Phase 2 baseline so
the regression figures stay comparable; the split column is part of the output path, so these
cannot collide with existing runs.

**This kernel only trains.** The stress test is CPU inference over a few thousand pairs and
runs locally against the pulled checkpoints. That keeps `src/eval_mutants.py` out of the
attached dataset, which matters right now because another kernel is reading that dataset and
re-versioning it underneath a running kernel is a known hazard in this project.

Two measurement details the local evaluator handles, recorded here because they decide how the
output should be read.

**Truncation.** Protein tokens are capped at 1200 residues, so an edit past position 1200
yields a token sequence identical to the wild type's. Three mutants are affected:
LRRK2(G2019S), MET(M1250T) and MET(Y1235D), whose full sequences run to 2527, 1390 and 1390
residues. The model cannot distinguish them from wild type by construction, and they are
detected by comparing token arrays rather than by the mutant table's `n_edits`, which counts
edits in the untruncated sequence and so reports all three as edited. They serve as a control:
the predicted shift must be exactly zero, which it is. Their measured shifts have a spread of
0.305 pKd, so this is real signal the model structurally cannot reach.

**Censoring.** DAVIS is left-censored at 10,000 nM, which is pKd 5.0, and 1788 of the 2992
triples have at least one measurement on that floor. A floor-to-floor pair has a measured
shift of exactly zero for reporting reasons rather than biological ones, so the headline
stratum is visible mutants with both measurements uncensored.

## What the output answers

Per seed, the ordinary `split_mutant` regression metrics on the test set, including the
uncensored subset, which say whether the model is competent on this split at all before any
shift analysis is read. The shift metrics themselves come from the local evaluator afterwards.

Expect roughly 40 minutes: 7.74 s per epoch measured for the DAVIS baseline, with runs
stopping somewhere between epoch 45 and 100 under `patience` 15.

Outputs survive only if this version is saved.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, SC4001_OUTPUT_ROOT="/kaggle/working/outputs", PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# Phase 3 needs the external kinase pool and the extra split columns, not just the Phase 2 tables.
required = ["external_kinases.csv", "external_tokens_davis.npz", "external_tokens_kiba.npz",
            "davis_extra_splits.csv", "kiba_extra_splits.csv"]
present = {p.name for p in (project / "data" / "processed").iterdir()}
missing = [name for name in required if name not in present and name + ".gz" not in present]
if missing:
    raise SystemExit("Dataset is missing Phase 3 inputs: " + str(missing))
print("phase 3 inputs OK")


In [ ]:
# Smoke the entry point on the split being run before spending quota, so a loader or
# split-resolution fault costs seconds rather than an hour.
smoke = subprocess.run(
    [sys.executable, "src/train.py", "--dataset", "davis",
     "--split", "split_mutant", "--smoke"],
    cwd=project, env=env, check=False,
)
if smoke.returncode != 0:
    raise SystemExit("Smoke run of src/train.py failed with code " + str(smoke.returncode)
                     + "; not starting the grid.")
for stale in Path("/kaggle/working/outputs").rglob("*-smoke"):
    shutil.rmtree(stale, ignore_errors=True)
print("smoke OK")


In [ ]:
# Phase 4: train on split_mutant so the mutant pairs are genuinely held out.
#
# Under split_cold_target every wild-type/mutant pair falls in train or val, never test,
# because mutants share a gene with their wild type and the split is by gene. So a cold-target
# checkpoint has already seen the pairs the stress test asks about. split_mutant is built for
# this: all 2992 mutant-target pairs are its test set, wild types are train and val.
#
# This kernel only TRAINS. The stress test itself is CPU inference over a few thousand pairs
# and runs locally against the pulled checkpoints, so `src/eval_mutants.py` does not need to
# be in the attached dataset and the dataset does not need re-versioning while another kernel
# is reading it.
#
# Default patience (15) and default output root, matching every other Phase 2 baseline so the
# regression numbers stay comparable. The split column is part of the output path.
GRID = [("split_mutant", seed) for seed in (4001, 4002, 4003)]

started = time.time()
status = []
for split, seed in GRID:
    t0 = time.time()
    label = "baseline davis " + split + " seed" + str(seed)
    print()
    print("===== " + label + " =====", flush=True)
    run = subprocess.run(
        [sys.executable, "src/train.py", "--dataset", "davis", "--split", split,
         "--seed", str(seed), "--resume"],
        cwd=project, env=env, check=False,
    )
    wall = time.time() - t0
    status.append((label, run.returncode, wall))
    print("----- rc=" + str(run.returncode) + " wall=" + format(wall / 60, ".1f")
          + " min -----", flush=True)

print()
print("Grid wall time", format((time.time() - started) / 60, ".1f"), "min")
for label, rc, wall in status:
    print("%-50s rc=%d %6.1f min" % (label, rc, wall / 60))
failed = [row for row in status if row[1] != 0]
print("FAILED RUNS:", failed if failed else "none")

# Regression quality on the split_mutant test set, which is the mutant pairs. This is not the
# stress test, it is the ordinary metric for this split, and it says whether the model is
# competent here at all before any shift analysis is read.
print()
print("split_mutant regression metrics (test set = the 2992 mutant pairs):")
for path in sorted(Path("/kaggle/working/outputs").glob("phase2/davis/split_mutant/seed*/metrics.json")):
    payload = json.loads(path.read_text())
    print("  seed%d  test MSE %.4f  CI %.4f  best_ep %3d  stop_ep %3d"
          % (payload["seed"], payload["test"]["all"]["mse"], payload["test"]["all"]["ci"],
             payload["best_epoch"], payload["stopped_epoch"]))
    uncensored = payload["test"].get("uncensored")
    if uncensored:
        print("         uncensored subset: n %d  MSE %.4f  CI %.4f"
              % (uncensored["n"], uncensored["mse"], uncensored["ci"]))
print()
print("Pull best.pt from each seed directory, then locally:")
print("  uv run python -m src.eval_mutants --run outputs/phase2/davis/split_mutant/seed4001 "
      "--run ...seed4002 --run ...seed4003")


In [ ]:
# Print every metrics.json so the numbers live in the saved notebook even if files are lost.
found = sorted(Path("/kaggle/working/outputs").glob("**/metrics.json"))
print(len(found), "metrics files")
rows = []
for path in found:
    payload = json.loads(path.read_text())
    print("=" * 70)
    print(path)
    print(json.dumps(payload, indent=2))
    model = payload["test"]["all"]
    phase = "phase3" if "phase3" in str(path) else "phase2"
    rows.append((phase, payload["split"], payload["seed"], payload.get("lambda_max"),
                 model["mse"], model["ci"], payload["best_epoch"], payload["total_seconds"]))

print("=" * 100)
print("%-7s %-26s %5s %7s %9s %8s %8s %9s"
      % ("phase", "split", "seed", "lambda", "mse", "ci", "best_ep", "minutes"))
for phase, split, seed, lam, mse, ci, best_ep, secs in sorted(rows):
    lam_text = "    n/a" if lam is None else "%7.2f" % lam
    print("%-7s %-26s %5d %s %9.4f %8.4f %8d %9.1f"
          % (phase, split, seed, lam_text, mse, ci, best_ep, secs / 60))
